In [1]:
import os
import requests

from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print("API key loaded:", GROQ_API_KEY is not None)

API key loaded: True


In [2]:
import os
import asyncio
import requests

from dotenv import load_dotenv

from openai import AsyncOpenAI

from semantic_kernel import Kernel
from semantic_kernel.functions import kernel_function
from semantic_kernel.agents import ChatCompletionAgent
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion


load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
MODEL = os.getenv("MODEL", "openai/gpt-oss-120b")

if not GROQ_API_KEY:
    raise ValueError(
        "GROQ_API_KEY not found. Please add it to your .env file."
    )

In [3]:
def get_coordinates(city: str):

    url = "https://geocoding-api.open-meteo.com/v1/search"

    params = {
        "name": city,
        "count": 1,
        "language": "en",
        "format": "json"
    }

    response = requests.get(
        url,
        params=params,
        timeout=10
    )

    response.raise_for_status()

    data = response.json()

    if "results" not in data:
        return None

    location = data["results"][0]

    return {
        "name": location["name"],
        "latitude": location["latitude"],
        "longitude": location["longitude"]
    }


def fetch_weather(city: str):

    location = get_coordinates(city)

    if location is None:
        return {
            "error": f"Could not find the city: {city}"
        }

    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": location["latitude"],
        "longitude": location["longitude"],
        "current": (
            "temperature_2m,"
            "relative_humidity_2m,"
            "weather_code"
        )
    }

    response = requests.get(
        url,
        params=params,
        timeout=10
    )

    response.raise_for_status()

    data = response.json()

    current = data["current"]

    return {
        "city": location["name"],
        "temperature": current["temperature_2m"],
        "humidity": current["relative_humidity_2m"],
        "weather_code": current["weather_code"]
    }


In [4]:
class WeatherPlugin:

    @kernel_function(
        name="get_weather",
        description=(
            "Get the current weather information for a city. "
            "Use this function whenever the user asks about "
            "current weather, temperature, humidity, or weather "
            "conditions."
        )
    )
    def get_weather(self, city: str) -> str:

        try:

            result = fetch_weather(city)

            if "error" in result:
                return result["error"]

            return (
                f"City: {result['city']}\n"
                f"Temperature: {result['temperature']} °C\n"
                f"Humidity: {result['humidity']}%\n"
                f"Weather code: {result['weather_code']}"
            )

        except requests.RequestException as e:

            return f"Weather API error: {str(e)}"



In [5]:
kernel = Kernel()

async_client = AsyncOpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)


chat_service = OpenAIChatCompletion(
    ai_model_id=MODEL,
    service_id="groq",
    async_client=async_client
)

kernel.add_service(chat_service)

kernel.add_plugin(
    WeatherPlugin(),
    plugin_name="Weather"
)

agent = ChatCompletionAgent(
    kernel=kernel,
    name="WeatherAgent",

    instructions="""
    You are a helpful weather assistant.

    You have access to a weather tool.

    Whenever the user asks about current weather,
    temperature, humidity, or weather conditions,
    use the Weather plugin.

    Do not invent current weather information.

    Always use the tool when current weather information
    is requested.

    After receiving the tool result, explain the answer
    clearly and briefly.
    """
)

In [6]:

async def main():

    print("=" * 60)
    print("        SEMANTIC KERNEL WEATHER AGENT")
    print("=" * 60)

    print("\nType 'exit' to stop.\n")

    while True:

        user_input = input("You: ")

        if user_input.lower() == "exit":
            print("Agent: Goodbye!")
            break

        try:

            response = await agent.get_response(
                messages=user_input
            )

            print("\nAgent:", response.content)
            print()

        except Exception as e:

            print("\nError:", e)
            print()

await main()

        SEMANTIC KERNEL WEATHER AGENT

Type 'exit' to stop.


Agent: The current temperature in Mumbai is **27.8 °C**.

Agent: Goodbye!
